# seq2ribo: Predict Ribosome Profiles from RNA Sequence

**seq2ribo** predicts where ribosomes sit on an mRNA transcript using only the RNA sequence as input. It combines a **structure-aware stochastic simulation (sTASEP)** of ribosome traffic with a **neural network polisher (Mamba)** to produce per-codon ribosome occupancy profiles, translation efficiency (TE) estimates, and protein expression predictions.

### Pipeline overview

```
RNA sequence
    │
    ├── ViennaRNA folding → secondary structure → per-codon geometry features
    │
    ├── sTASEP simulation  → physics-based ribosome occupancy estimate
    │
    └── Mamba polisher     → refined per-codon ribosome counts
                                │
                                ├── A-site profile (riboseq task)
                                ├── Translation efficiency (TE task)
                                └── Protein expression (protein task)
```

**Supported cell lines:** HEK293, LCL, RPE-1, iPSC

**Paper:** [bioRxiv](https://www.biorxiv.org/content/10.64898/2026.02.08.700508v2)  
**Repository:** [github.com/Kingsford-Group/seq2ribo](https://github.com/Kingsford-Group/seq2ribo)

---
## 1. Setup

Import the library and initialize the predictor. Make sure you have followed the [installation instructions](../INSTALL.md) first.

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

sys.path.insert(0, str(Path.cwd().parent))
from seq2ribo import Seq2Ribo

WEIGHTS_DIR = str(Path.cwd().parent / "weights")

# Publication-quality plot defaults
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.figsize": (12, 4),
})

COLORS = {
    "ground_truth": "#2d2d2d",
    "stasep": "#3b82f6",
    "polisher": "#ef4444",
    "polisher_fill": "#fca5a5",
    "stasep_fill": "#93c5fd",
}

print("Setup complete.")

In [ ]:
# Load sample test-set transcripts (real ribo-seq data)
with open("sample_data/demo_transcripts.pkl", "rb") as f:
    demo_data = pickle.load(f)

print(f"Loaded {len(demo_data)} demo transcripts:")
for key in sorted(demo_data.keys()):
    d = demo_data[key]
    n_codons = len(d["sequence"]) // 3
    print(f"  {d['cell_line']:5s}  {d['tx_id']:25s}  {n_codons:4d} codons")

---
## 2. Ribosome A-site Profile Prediction

The core task: given an mRNA coding sequence, predict the per-codon ribosome occupancy (A-site profile).

We compare three signals:
- **Ground truth** — experimentally measured A-site counts from ribo-seq
- **sTASEP only** — physics-based simulation without neural refinement
- **sTASEP + Polisher** — simulation refined by the Mamba neural network

In [ ]:
# Initialize iPSC predictor
predictor_ipsc = Seq2Ribo(cell_line="ipsc", weights_dir=WEIGHTS_DIR)

In [ ]:
# Pick a medium-length transcript from the iPSC test set
sample = demo_data[("ipsc", "ENST00000367883.3")]
seq = sample["sequence"]
n_codons = len(seq) // 3

# Ground truth: convert nucleotide-level counts to per-codon by summing each triplet
gt_asite_nuc = sample["asite_counts"]
gt_asite = gt_asite_nuc.reshape(-1, 3).sum(axis=1).astype(np.float64)

print(f"Transcript: {sample['tx_id']}")
print(f"Length: {n_codons} codons ({len(seq)} nt)")
print(f"Total A-site reads: {int(gt_asite.sum())}")

In [ ]:
# Predict: sTASEP only (no neural network)
pred_stasep = predictor_ipsc.predict(
    seq, task="riboseq", use_polisher=False, n_stasep_runs=50
)[0]

# Predict: sTASEP + Polisher (full model)
pred_polished = predictor_ipsc.predict(
    seq, task="riboseq", use_polisher=True, n_stasep_runs=50
)[0]

print(f"sTASEP prediction shape: {pred_stasep.shape}")
print(f"Polished prediction shape: {pred_polished.shape}")

In [ ]:
# Normalize for visual comparison: scale predictions to match ground-truth total
gt_total = gt_asite.sum()
stasep_scaled = pred_stasep * (gt_total / (pred_stasep.sum() + 1e-9))
polished_scaled = pred_polished * (gt_total / (pred_polished.sum() + 1e-9))

x = np.arange(n_codons)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Ground truth
axes[0].fill_between(x, gt_asite, alpha=0.4, color=COLORS["ground_truth"])
axes[0].plot(x, gt_asite, color=COLORS["ground_truth"], linewidth=0.8)
axes[0].set_ylabel("A-site counts")
axes[0].set_title(f"Ground Truth Ribo-seq  —  {sample['tx_id']} (iPSC)")

# sTASEP only
axes[1].fill_between(x, stasep_scaled, alpha=0.3, color=COLORS["stasep_fill"])
axes[1].plot(x, stasep_scaled, color=COLORS["stasep"], linewidth=0.8)
r_st, _ = pearsonr(gt_asite, stasep_scaled)
axes[1].set_ylabel("Predicted counts")
axes[1].set_title(f"sTASEP Simulation Only  (Pearson r = {r_st:.3f})")

# Polished
axes[2].fill_between(x, polished_scaled, alpha=0.3, color=COLORS["polisher_fill"])
axes[2].plot(x, polished_scaled, color=COLORS["polisher"], linewidth=0.8)
r_pol, _ = pearsonr(gt_asite, polished_scaled)
axes[2].set_ylabel("Predicted counts")
axes[2].set_title(f"sTASEP + Mamba Polisher  (Pearson r = {r_pol:.3f})")
axes[2].set_xlabel("Codon position")

for ax in axes:
    ax.set_xlim(0, n_codons - 1)
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# Overlay comparison
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(x, gt_asite, color=COLORS["ground_truth"], linewidth=1.2, label="Ground truth", alpha=0.8)
ax.plot(x, stasep_scaled, color=COLORS["stasep"], linewidth=1.0, label=f"sTASEP only (r={r_st:.3f})", alpha=0.7)
ax.plot(x, polished_scaled, color=COLORS["polisher"], linewidth=1.0, label=f"sTASEP + Polisher (r={r_pol:.3f})", alpha=0.7)

ax.set_xlabel("Codon position")
ax.set_ylabel("A-site counts (scaled)")
ax.set_title(f"Ribosome Profile Comparison — {sample['tx_id']} (iPSC)")
ax.legend(loc="upper right", framealpha=0.9)
ax.set_xlim(0, n_codons - 1)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

---
## 3. Cell-line Comparison

The same gene can show different ribosome occupancy patterns across cell types due to differences in tRNA availability, codon usage bias, and regulatory context. seq2ribo captures this through cell-line-specific sTASEP rate parameters.

In [ ]:
# Initialize LCL predictor
predictor_lcl = Seq2Ribo(cell_line="lcl", weights_dir=WEIGHTS_DIR)

In [ ]:
# Use the same iPSC transcript sequence
pred_ipsc = predictor_ipsc.predict(seq, task="riboseq", use_polisher=True, n_stasep_runs=50)[0]
pred_lcl = predictor_lcl.predict(seq, task="riboseq", use_polisher=True, n_stasep_runs=50)[0]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

cell_info = [
    ("iPSC", pred_ipsc, "#3b82f6", "#93c5fd"),
    ("LCL", pred_lcl, "#8b5cf6", "#c4b5fd"),
]

for ax, (name, pred, color, fill) in zip(axes, cell_info):
    ax.fill_between(x, pred, alpha=0.3, color=fill)
    ax.plot(x, pred, color=color, linewidth=0.9)
    ax.set_ylabel("Predicted counts")
    ax.set_title(f"{name} — {sample['tx_id']}")
    ax.set_xlim(0, n_codons - 1)
    ax.spines[["top", "right"]].set_visible(False)

axes[1].set_xlabel("Codon position")
fig.suptitle("Cell-line-specific Ribosome Profiles for the Same Transcript", fontsize=15, y=1.02)
fig.tight_layout()
plt.show()

---
## 4. Translation Efficiency (TE) — CDS Only

Translation efficiency measures how effectively an mRNA is translated into protein, relative to its abundance. seq2ribo predicts TE as a scalar value from the coding sequence (CDS).

The model outputs a value in [0, 1] which is then inverse-transformed back to the original TE scale.

In [ ]:
# Collect CDS sequences from all iPSC demo transcripts
ipsc_transcripts = [
    demo_data[k] for k in sorted(demo_data.keys()) if k[0] == "ipsc"
]
tx_names = [d["tx_id"] for d in ipsc_transcripts]
tx_seqs = [d["sequence"] for d in ipsc_transcripts]

# Predict TE (inverse-transformed, real TE scale)
te_values = predictor_ipsc.predict(tx_seqs, task="te", n_stasep_runs=50)

# Also get scaled [0,1] values
te_scaled = predictor_ipsc.predict(tx_seqs, task="te", n_stasep_runs=50, return_scaled_te=True)

print(f"{'Transcript':<28s} {'TE (original scale)':>20s} {'TE (scaled 0-1)':>18s}")
print("-" * 70)
for name, te, te_s in zip(tx_names, te_values, te_scaled):
    print(f"{name:<28s} {te:>20.4f} {te_s:>18.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

short_names = [n.split(".")[0].replace("ENST", "") for n in tx_names]
bars = ax.bar(short_names, te_values, color="#3b82f6", edgecolor="#1e40af", linewidth=0.8)

for bar, val in zip(bars, te_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02 * max(te_values),
            f"{val:.2f}", ha="center", va="bottom", fontsize=11)

ax.set_xlabel("Transcript")
ax.set_ylabel("Translation Efficiency")
ax.set_title("Predicted Translation Efficiency (CDS only) — iPSC")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

---
## 5. Translation Efficiency (TE) — CDS + UTR Context

UTR sequences (5′ and 3′ untranslated regions) influence translation initiation and stability. seq2ribo can incorporate UTR context for a more complete TE prediction.

In UTR-aware mode, you provide the 5′ UTR, CDS, and 3′ UTR as separate inputs.

In [ ]:
# Example UTR sequences (representative lengths)
utr5_seqs = [
    "GCUCUCCGACUCCGGCGCGGCGCCCCGACUCCGGUACAGCCGUUCCGCCCACGCUCCGAC",
    "AGACCUGGCUGCUCUGAACAGCUCCGGCCUUA",
    "CCCGCCGGAGCCAUGUCUAUCAUGC",
]
cds_seqs = [d["sequence"] for d in ipsc_transcripts]
utr3_seqs = [
    "CCAGCUCCUGGUACCCUGGCACCUGUAAUAAUUUGUGUUCCUUCUAAAAGGUAAA",
    "UUAUCUAAAUUGUACAAAGGUAAUAAACUGUUGUUUCUAAUAAAUACAGACUAUUU",
    "CCUAAUAAAGAUCACUUUUAUAUCAAAUGUUUCA",
]

# CDS-only TE
te_cds = predictor_ipsc.predict(cds_seqs, task="te", n_stasep_runs=50)

# CDS + UTR TE
te_utr = predictor_ipsc.predict(
    task="te", use_utr=True,
    utr5_list=utr5_seqs, cds_list=cds_seqs, utr3_list=utr3_seqs,
    n_stasep_runs=50,
)

print(f"{'Transcript':<28s} {'TE (CDS)':>12s} {'TE (CDS+UTR)':>14s}")
print("-" * 58)
for name, t_cds, t_utr in zip(tx_names, te_cds, te_utr):
    print(f"{name:<28s} {t_cds:>12.4f} {t_utr:>14.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

bar_width = 0.35
indices = np.arange(len(tx_names))

ax.bar(indices - bar_width / 2, te_cds, bar_width, label="CDS only",
       color="#3b82f6", edgecolor="#1e40af", linewidth=0.8)
ax.bar(indices + bar_width / 2, te_utr, bar_width, label="CDS + UTR",
       color="#10b981", edgecolor="#065f46", linewidth=0.8)

ax.set_xticks(indices)
ax.set_xticklabels(short_names)
ax.set_xlabel("Transcript")
ax.set_ylabel("Translation Efficiency")
ax.set_title("CDS-only vs CDS+UTR Translation Efficiency — iPSC")
ax.legend(framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

---
## 6. Protein Expression Prediction

seq2ribo can predict relative protein expression levels from the CDS. The model uses **Monte Carlo dropout** (32 stochastic forward passes) to produce a robust estimate.

The output is a scalar proportional to expected protein abundance.

In [ ]:
# Protein expression prediction
expr_values = predictor_ipsc.predict(tx_seqs, task="protein", n_stasep_runs=50)

print(f"{'Transcript':<28s} {'Protein Expression':>20s}")
print("-" * 50)
for name, expr in zip(tx_names, expr_values):
    print(f"{name:<28s} {expr:>20.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(short_names, expr_values, color="#f59e0b", edgecolor="#b45309", linewidth=0.8)

for bar, val in zip(bars, expr_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02 * max(expr_values),
            f"{val:.3f}", ha="center", va="bottom", fontsize=11)

ax.set_xlabel("Transcript")
ax.set_ylabel("Predicted Expression")
ax.set_title("Predicted Protein Expression — iPSC")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

---
## 7. Try Your Own Sequence

Paste an RNA coding sequence (CDS) below to generate predictions. The sequence must:
- Start with a start codon (AUG)
- End with a stop codon (UAA, UAG, or UGA)
- Have a length that is a multiple of 3 (complete codons)
- Use RNA alphabet (A, U, G, C) — T is automatically converted to U

In [ ]:
# === PASTE YOUR RNA CDS SEQUENCE HERE ===
my_sequence = "AUGGCCAAGCUGAAGAAGGCCACCUUCGGCUUCAGCGAGGGCACCAAGGUGCUGUUCC" \
              "AGCCAUCCAUGUCCGUGUCCUUCGACAAGUUCGAGAAGGACGGCGACAUGGCCAUGGA" \
              "GUCCCUGAAGGCCAUCUGCACCGACAACUUCUCCUGGAACCACCUGGUGGAGCUGCAC" \
              "GGCAAGCAGGAGUUCGAGAGCAUGGGCGCCGACUGGAAUAACAUUGCUAAGAUCAUCU" \
              "UCGAGGGCCUCGGCUUCAAGAAGGACGGCAACCCAAUCCCCAACCCAGAGCUCAAGGCC" \
              "AUCGGCAAGAAGGUGGGCCCUGACAACGACCAGUAA"

cell_line = "ipsc"  # Change to: "hek293", "lcl", "rpe", or "ipsc"

# ==========================================

predictor = Seq2Ribo(cell_line=cell_line, weights_dir=WEIGHTS_DIR)

# Run all three prediction tasks
profile_stasep = predictor.predict(my_sequence, task="riboseq", use_polisher=False, n_stasep_runs=50)[0]
profile_polished = predictor.predict(my_sequence, task="riboseq", use_polisher=True, n_stasep_runs=50)[0]
te_value = predictor.predict(my_sequence, task="te", n_stasep_runs=50)[0]
expr_value = predictor.predict(my_sequence, task="protein", n_stasep_runs=50)[0]

n_cod = len(my_sequence.replace("T", "U").replace("t", "u")) // 3
print(f"Sequence length: {n_cod} codons")
print(f"Translation Efficiency: {te_value:.4f}")
print(f"Protein Expression:     {expr_value:.4f}")

In [ ]:
# Full visualization panel for your sequence
fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], hspace=0.35, wspace=0.3)

# Top: A-site profile
ax_profile = fig.add_subplot(gs[0, :])
xp = np.arange(n_cod)
ax_profile.plot(xp, profile_stasep, color=COLORS["stasep"], linewidth=1.0, label="sTASEP only", alpha=0.7)
ax_profile.plot(xp, profile_polished, color=COLORS["polisher"], linewidth=1.0, label="sTASEP + Polisher", alpha=0.8)
ax_profile.fill_between(xp, profile_polished, alpha=0.15, color=COLORS["polisher"])
ax_profile.set_xlabel("Codon position")
ax_profile.set_ylabel("Predicted ribosome count")
ax_profile.set_title(f"Predicted A-site Profile ({cell_line.upper()})")
ax_profile.legend(loc="upper right", framealpha=0.9)
ax_profile.set_xlim(0, n_cod - 1)
ax_profile.spines[["top", "right"]].set_visible(False)

# Bottom left: TE
ax_te = fig.add_subplot(gs[1, 0])
ax_te.barh(["TE"], [te_value], color="#3b82f6", edgecolor="#1e40af", height=0.5)
ax_te.set_xlabel("Translation Efficiency")
ax_te.set_title("Predicted TE")
ax_te.text(te_value + 0.01 * abs(te_value), 0, f"{te_value:.3f}", va="center", fontsize=12)
ax_te.spines[["top", "right"]].set_visible(False)

# Bottom right: Protein expression
ax_expr = fig.add_subplot(gs[1, 1])
ax_expr.barh(["Expression"], [expr_value], color="#f59e0b", edgecolor="#b45309", height=0.5)
ax_expr.set_xlabel("Protein Expression")
ax_expr.set_title("Predicted Protein Expression")
ax_expr.text(expr_value + 0.01 * abs(expr_value), 0, f"{expr_value:.3f}", va="center", fontsize=12)
ax_expr.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"seq2ribo Full Prediction Panel ({n_cod} codons, {cell_line.upper()})", fontsize=15, y=1.02)
plt.show()

---
## 8. Batch Prediction from FASTA

Process multiple sequences at once by reading from a FASTA file. This is useful for screening libraries or comparing transcript variants.

In [ ]:
def read_fasta(path):
    """Simple FASTA reader returning list of (header, sequence) tuples."""
    sequences = []
    header, seq_parts = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header is not None:
                    sequences.append((header, "".join(seq_parts)))
                header = line[1:].split()[0]
                seq_parts = []
            elif line:
                seq_parts.append(line)
    if header is not None:
        sequences.append((header, "".join(seq_parts)))
    return sequences

# Create a small example FASTA
example_fasta = Path("sample_data/example_batch.fa")
with open(example_fasta, "w") as f:
    for d in ipsc_transcripts:
        f.write(f">{d['tx_id']}\n")
        seq = d["sequence"]
        for i in range(0, len(seq), 80):
            f.write(seq[i:i+80] + "\n")

print(f"Wrote {len(ipsc_transcripts)} sequences to {example_fasta}")

In [ ]:
# Read FASTA and run batch predictions
fasta_seqs = read_fasta(example_fasta)
headers = [h for h, _ in fasta_seqs]
sequences = [s for _, s in fasta_seqs]

batch_te = predictor_ipsc.predict(sequences, task="te", n_stasep_runs=50)
batch_expr = predictor_ipsc.predict(sequences, task="protein", n_stasep_runs=50)

print(f"{'Transcript':<28s} {'Codons':>8s} {'TE':>10s} {'Expression':>12s}")
print("=" * 62)
for header, seq, te, expr in zip(headers, sequences, batch_te, batch_expr):
    n = len(seq) // 3
    print(f"{header:<28s} {n:>8d} {te:>10.4f} {expr:>12.4f}")

---

## Summary

| Task | Input | Output | Use case |
|------|-------|--------|----------|
| `riboseq` | CDS sequence | Per-codon ribosome counts | Identify ribosome pause sites, stalling |
| `te` | CDS (or CDS + UTRs) | Translation efficiency scalar | Compare translational output across variants |
| `protein` | CDS sequence | Protein expression scalar | Predict relative protein levels |

For more details, see the [paper](https://www.biorxiv.org/content/10.64898/2026.02.08.700508v2) and [README](../README.md).